# Decision Tree Optimization

In [138]:
import pandas as pd
import json

In [139]:
df = pd.DataFrame({'studied': [1,2,3,4,5,6,7,8,9,10,11,12], 
        'passed': [0,0,0,0,1,1,1,1,1,1,1,1]})

In [140]:
df

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


## Mid point

In [141]:
# compute mid points between values of studied
mid_points = []
for i in range(len(df['studied']) - 1):
    mid_point = (df['studied'][i] + df['studied'][i+1]) / 2
    # [i] refer to the fist value and [i+1] refer to the second value and so on
    print(mid_point)
    mid_points.append(mid_point)

1.5
2.5
3.5
4.5
5.5
6.5
7.5
8.5
9.5
10.5
11.5


In [142]:
x = df['studied'].values
x

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])

In [143]:
print("from 1st item to 2nd last:")
print(x[:-1])
print("from 2nd item to last:")
print(x[1:])

from 1st item to 2nd last:
[ 1  2  3  4  5  6  7  8  9 10 11]
from 2nd item to last:
[ 2  3  4  5  6  7  8  9 10 11 12]


In [144]:
mid_points = (x[:-1] + x[1:]) / 2
print(mid_points)

[ 1.5  2.5  3.5  4.5  5.5  6.5  7.5  8.5  9.5 10.5 11.5]


## Sliding Windows

### Gini Formula

In [145]:
def gini(y):
    if len(y) == 0:
        return 0
    p0 = (y == 0).sum() / len(y)
    p1 = (y == 1).sum() / len(y)
    gini = 1 - (p0**2 + p1**2)
    return gini

In [146]:
# test gini
gini(df['passed'].values)

np.float64(0.4444444444444444)

In [147]:
def weighted_gini(y_left, y_right):
    n = len(y_left) + len(y_right)
    gini_left = gini(y_left)
    gini_right = gini(y_right)
    weighted_gini = (len(y_left) / n) * gini_left + (len(y_right) / n) * gini_right
    return weighted_gini

In [148]:
# Split base on first mid point
mid_point = mid_points[0]
left_node = df[df['studied'] <= mid_point]
right_node = df[df['studied'] > mid_point]
weighted_gini_first = weighted_gini(left_node['passed'].values, right_node['passed'].values)

In [149]:
print(mid_point)
print(left_node)
print(right_node)

1.5
   studied  passed
0        1       0
    studied  passed
1         2       0
2         3       0
3         4       0
4         5       1
5         6       1
6         7       1
7         8       1
8         9       1
9        10       1
10       11       1
11       12       1


In [150]:
# second mid point split
mid_point = mid_points[1]
left_node = df[df['studied'] <= mid_point]
right_node = df[df['studied'] > mid_point]
print(mid_point)
print(left_node)
print(right_node)
weighted_gini_second = weighted_gini(left_node['passed'].values, right_node['passed'].values)


2.5
   studied  passed
0        1       0
1        2       0
    studied  passed
2         3       0
3         4       0
4         5       1
5         6       1
6         7       1
7         8       1
8         9       1
9        10       1
10       11       1
11       12       1


In [151]:
print("Weighted Gini for first split:", weighted_gini_first)
print("Weighted Gini for second split:", weighted_gini_second)

Weighted Gini for first split: 0.36363636363636365
Weighted Gini for second split: 0.26666666666666655


## Sliding Windows

Pseudo code:

1. Sort feature
2. Initialize LEFT counts to zero
3. Initialize RIGHT counts to total class counts
4. Move one sample at a time from RIGHT to LEFT
5. Update counts
6. Compute weighted Gini
7. Track best boundary
8. Convert best boundary to midpoint threshold




In [152]:
import numpy as np

In [153]:
def gini(no_negative, no_positive):
    total = no_negative + no_positive
    if total == 0:
        return 0

    p0 = no_negative/total
    p1 = no_positive/total
    gini = 1 - (p0**2 + p1**2)
    return gini

In [154]:
def best_gini_search(df):

    best_gini = float('inf')
    best_split_value = None

    # Sort features
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values

    # initialized count
    left_p0_count = 0
    left_p1_count = 0
    right_p0_count = (y==0).sum()
    right_p1_count = (y==1).sum()

    # print(left_p0_count)
    # print(left_p1_count)
    # print(right_p0_count)
    # print(right_p1_count)

    # initializa total number of data
    n = len(y)
    n_right = n
    n_left = 0

    for i in range(n-1):
        if y[i] == 0:
            left_p0_count += 1
            right_p0_count -= 1
            n_left += 1
            n_right -= 1
        else:
            left_p1_count += 1
            right_p1_count -= 1
            n_left += 1
            n_right -= 1

        print('left false',left_p0_count)
        print('left, true',left_p1_count)
        print('total left',n_left)
        print('right false',right_p0_count)
        print('right, true',right_p1_count)
        print('total right',n_right)
        
        left_gini = gini(left_p0_count, left_p1_count)
        right_gini = gini(right_p0_count, right_p1_count)
        weighted_gini = (n_left / n) *  left_gini + (n_right / n) * right_gini
        print('weighted gini',weighted_gini)
 
        if weighted_gini < best_gini:
            best_gini = weighted_gini
            best_split_value = (x[i] + x[i+1])/2
            #print(best_split_value)
            print('best gini', best_gini)
        
    return best_split_value, best_gini


In [155]:
best_gini_search(df)

left false 1
left, true 0
total left 1
right false 3
right, true 8
total right 11
weighted gini 0.36363636363636365
best gini 0.36363636363636365
left false 2
left, true 0
total left 2
right false 2
right, true 8
total right 10
weighted gini 0.26666666666666655
best gini 0.26666666666666655
left false 3
left, true 0
total left 3
right false 1
right, true 8
total right 9
weighted gini 0.14814814814814814
best gini 0.14814814814814814
left false 4
left, true 0
total left 4
right false 0
right, true 8
total right 8
weighted gini 0.0
best gini 0.0
left false 4
left, true 1
total left 5
right false 0
right, true 7
total right 7
weighted gini 0.13333333333333328
left false 4
left, true 2
total left 6
right false 0
right, true 6
total right 6
weighted gini 0.2222222222222222
left false 4
left, true 3
total left 7
right false 0
right, true 5
total right 5
weighted gini 0.2857142857142858
left false 4
left, true 4
total left 8
right false 0
right, true 4
total right 4
weighted gini 0.3333333333

(np.float64(4.5), np.float64(0.0))

## CUMSUM

In [156]:
y = df['passed'].values
x = df['studied'].values

In [157]:
left_negative = (y == 0).cumsum()
left_positive = (y == 1).cumsum()
print('Left Negative', left_negative)
print('Left Positive', left_positive)

Left Negative [1 2 3 4 4 4 4 4 4 4 4 4]
Left Positive [0 0 0 0 1 2 3 4 5 6 7 8]


In [158]:
negative_total = left_negative[-1]
positive_total = left_positive[-1]
print('Negative Total', negative_total)
print('Positive Total', positive_total)

Negative Total 4
Positive Total 8


In [159]:
right_negative = negative_total - left_negative
right_positive = positive_total - left_positive
print('Right Negative', right_negative)
print('Right Positive', right_positive)

Right Negative [3 2 1 0 0 0 0 0 0 0 0 0]
Right Positive [8 8 8 8 7 6 5 4 3 2 1 0]


In [160]:
def best_gini_cumsum(df):

    best_gini = float('inf')
    best_split_value = None

    # Sort features
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values

    # initialized count
    left_negative = (y == 0).cumsum()
    left_positive = (y == 1).cumsum()
    negative_total = left_negative[-1]
    positive_total = left_positive[-1]
    right_negative = negative_total - left_negative
    right_positive = positive_total - left_positive


    # initialize total number of data
    n = len(y)
    n_right = n
    n_left = 0

    for i in range(n-1):
        n_left += 1
        n_right -= 1

        left_negative[i]
        
        left_gini = gini(left_negative[i], left_positive[i])
        right_gini = gini(right_negative[i], right_positive[i])
        weighted_gini = (n_left / n) *  left_gini + (n_right / n) * right_gini
        print('weighted gini',weighted_gini)
 
        if weighted_gini < best_gini:
            best_gini = weighted_gini
            best_split_value = (x[i] + x[i+1])/2
            #print(best_split_value)
            print('best gini', best_gini)
        
    return best_split_value, best_gini


In [161]:
df

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


In [162]:
split, best_gini = best_gini_cumsum(df)
print(f"Best gini {best_gini} and the threshold is {split}")

weighted gini 0.36363636363636365
best gini 0.36363636363636365
weighted gini 0.26666666666666655
best gini 0.26666666666666655
weighted gini 0.14814814814814814
best gini 0.14814814814814814
weighted gini 0.0
best gini 0.0
weighted gini 0.13333333333333328
weighted gini 0.2222222222222222
weighted gini 0.2857142857142858
weighted gini 0.3333333333333333
weighted gini 0.37037037037037035
weighted gini 0.4
weighted gini 0.42424242424242425
Best gini 0.0 and the threshold is 4.5


## Vectorization

In [163]:
left_negative = (y == 0).cumsum()
left_positive = (y == 1).cumsum()
print('Left Negative', left_negative)
print('Left Positive', left_positive)

Left Negative [1 2 3 4 4 4 4 4 4 4 4 4]
Left Positive [0 0 0 0 1 2 3 4 5 6 7 8]


In [164]:
left_count = left_negative + left_positive
left_count

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])

In [165]:
negative_total = left_negative[-1]
positive_total = left_positive[-1]
print('Negative Total', negative_total)
print('Positive Total', positive_total)

Negative Total 4
Positive Total 8


In [166]:
n = negative_total + positive_total
n

np.int64(12)

In [167]:
right_negative = negative_total - left_negative
right_positive = positive_total - left_positive
print('Right Negative', right_negative)
print('Right Positive', right_positive)

Right Negative [3 2 1 0 0 0 0 0 0 0 0 0]
Right Positive [8 8 8 8 7 6 5 4 3 2 1 0]


In [168]:
right_count = right_negative + right_positive
right_count

array([11, 10,  9,  8,  7,  6,  5,  4,  3,  2,  1,  0])

In [169]:
left_negative/left_count

array([1.        , 1.        , 1.        , 1.        , 0.8       ,
       0.66666667, 0.57142857, 0.5       , 0.44444444, 0.4       ,
       0.36363636, 0.33333333])

In [170]:
left_positive/left_count

array([0.        , 0.        , 0.        , 0.        , 0.2       ,
       0.33333333, 0.42857143, 0.5       , 0.55555556, 0.6       ,
       0.63636364, 0.66666667])

In [171]:
right_negative/right_count

/tmp/ipykernel_1001/1900881966.py:1: RuntimeWarning: invalid value encountered in divide
  right_negative/right_count


array([0.27272727, 0.2       , 0.11111111, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        ,        nan])

In [172]:
right_positive/right_count

/tmp/ipykernel_1001/1516977460.py:1: RuntimeWarning: invalid value encountered in divide
  right_positive/right_count


array([0.72727273, 0.8       , 0.88888889, 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        ,        nan])

In actual fact, we should compute n - 1. Therefore, we need to drop the last item 

In [173]:
left_negative = left_negative[:-1]
left_positive = left_positive[:-1]
left_count = left_negative + left_positive

In [174]:
right_negative = right_negative[:-1]
right_positive = right_positive[:-1]
right_count = right_negative + right_positive

In [175]:
def gini(no_negative: np.array, no_positive: np.array, no_total: np.array):
    p0 = no_negative/no_total
    p1 = no_positive/no_total
    gini = 1 - (p0**2 + p1**2)
    return gini

In [176]:
left_gini = gini(left_negative, left_positive, left_count)
right_gini = gini(right_negative, right_positive, right_count)
weighted_gini = (left_count / n) *  left_gini + (right_count / n) * right_gini

In [177]:
weighted_gini

array([0.36363636, 0.26666667, 0.14814815, 0.        , 0.13333333,
       0.22222222, 0.28571429, 0.33333333, 0.37037037, 0.4       ,
       0.42424242])

In [178]:
dashboard = np.c_[left_negative,left_positive,left_count,right_negative, right_positive, right_count,left_gini,right_gini, weighted_gini]

In [179]:
dashboard_df = pd.DataFrame(dashboard, columns=['left_negative','left_positive','left_count','right_negative', 'right_positive', 'right_count','left_gini','right_gini', 'weighted_gini'])

In [180]:
dashboard_df

,left_negative,left_positive,left_count,right_negative,right_positive,right_count,left_gini,right_gini,weighted_gini
0,1.0,0.0,1.0,3.0,8.0,11.0,0.000000,0.396694,0.363636
1,2.0,0.0,2.0,2.0,8.0,10.0,0.000000,0.320000,0.266667
2,3.0,0.0,3.0,1.0,8.0,9.0,0.000000,0.197531,0.148148
3,4.0,0.0,4.0,0.0,8.0,8.0,0.000000,0.000000,0.000000
4,4.0,1.0,5.0,0.0,7.0,7.0,0.320000,0.000000,0.133333
5,4.0,2.0,6.0,0.0,6.0,6.0,0.444444,0.000000,0.222222
6,4.0,3.0,7.0,0.0,5.0,5.0,0.489796,0.000000,0.285714
7,4.0,4.0,8.0,0.0,4.0,4.0,0.500000,0.000000,0.333333
8,4.0,5.0,9.0,0.0,3.0,3.0,0.493827,0.000000,0.370370
9,4.0,6.0,10.0,0.0,2.0,2.0,0.480000,0.000000,0.400000


In [181]:
index = dashboard_df['weighted_gini'].argmin()

In [182]:
x[index]

np.int64(4)

In [183]:
threshold = (x[index] + x[index+1])/2
threshold

np.float64(4.5)

In [184]:
def best_gini_vector(df):

    best_gini = float('inf')
    best_split_value = None

    # Sort features
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values

    # perform cumsum
    left_negative = (y == 0).cumsum()
    left_positive = (y == 1).cumsum()
    negative_total = left_negative[-1]
    positive_total = left_positive[-1]
    right_negative = negative_total - left_negative
    right_positive = positive_total - left_positive
    left_negative = left_negative[:-1]
    left_positive = left_positive[:-1]
    right_negative = right_negative[:-1]
    right_positive = right_positive[:-1]


    # initialize total number of data
    n = len(y)
    n_right = right_negative + right_positive
    n_left = left_negative + left_positive

    left_gini = gini(left_negative, left_positive, left_count)
    right_gini = gini(right_negative, right_positive, right_count)
    
    weighted_gini = (n_left / n) *  left_gini + (n_right / n) * right_gini
    print('weighted gini',weighted_gini)

    best_gini = weighted_gini.min()
    best_split_value = (x[weighted_gini.argmin()] + x[weighted_gini.argmin() + 1])/2

    return best_split_value, best_gini


In [185]:
split, best_gini = best_gini_vector(df)
print(f"Best gini {best_gini} and the threshold is {split}")

weighted gini [0.36363636 0.26666667 0.14814815 0.         0.13333333 0.22222222
 0.28571429 0.33333333 0.37037037 0.4        0.42424242]
Best gini 0.0 and the threshold is 4.5


## Cases with Identical Values

In [186]:
df2 = pd.DataFrame({'studied': [1,1,1,2,2,2,2,3,3,3,4,4], 
        'passed': [0,0,0,0,1,1,1,1,1,1,1,1]})

In [187]:
split, best_gini = best_gini_vector(df2)
print(f"Best gini {best_gini} and the threshold is {split}")

weighted gini [0.36363636 0.26666667 0.14814815 0.         0.13333333 0.22222222
 0.28571429 0.33333333 0.37037037 0.4        0.42424242]
Best gini 0.0 and the threshold is 2.0


In [188]:
x = df2['studied'].values
y = df2['passed'].values

In [189]:
valid = x[:-1] != x[1:]

In [190]:
valid

array([False, False,  True, False, False, False,  True, False, False,
        True, False])

In [191]:
x

array([1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 4, 4])

In [192]:
left_negative = (y == 0).cumsum()
left_positive = (y == 1).cumsum()
print('Left Negative', left_negative)
print('Left Positive', left_positive)

Left Negative [1 2 3 4 4 4 4 4 4 4 4 4]
Left Positive [0 0 0 0 1 2 3 4 5 6 7 8]


In [193]:
negative_total = left_negative[-1]
positive_total = left_positive[-1]
print('Negative Total', negative_total)
print('Positive Total', positive_total)

Negative Total 4
Positive Total 8


In [194]:
right_negative = negative_total - left_negative
right_positive = positive_total - left_positive
print('Right Negative', right_negative)
print('Right Positive', right_positive)

Right Negative [3 2 1 0 0 0 0 0 0 0 0 0]
Right Positive [8 8 8 8 7 6 5 4 3 2 1 0]


In [195]:
left_negative = left_negative[:-1]
left_positive = left_positive[:-1]
left_count = left_negative + left_positive

In [196]:
right_negative = right_negative[:-1]
right_positive = right_positive[:-1]
right_count = right_negative + right_positive

In [197]:
left_negative = left_negative[valid]
left_positive = left_positive[valid]

right_negative = right_negative[valid]
right_positive = right_positive[valid]

In [198]:
print('Left Negative', left_negative)
print('Left Positive', left_positive)
print('Right Negative', right_negative)
print('Right Positive', right_positive)

Left Negative [3 4 4]
Left Positive [0 3 6]
Right Negative [1 0 0]
Right Positive [8 5 2]


In [206]:
left_count = left_negative + left_positive
right_count = right_negative + right_positive
left_count

array([ 3,  7, 10])

In [200]:
left_gini = gini(left_negative, left_positive, left_count)
right_gini = gini(right_negative, right_positive, right_count)
weighted_gini = (left_count / n) *  left_gini + (right_count / n) * right_gini
weighted_gini

array([0.14814815, 0.28571429, 0.4       ])

In [216]:
def best_gini_CART_ready(df):

    best_gini = float('inf')
    best_split_value = None

    # Sort features
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values
    valid = x[:-1] != x[1:]

    # perform cumsum
    left_negative = (y == 0).cumsum()
    left_positive = (y == 1).cumsum()
    negative_total = left_negative[-1]
    positive_total = left_positive[-1]
    right_negative = negative_total - left_negative
    right_positive = positive_total - left_positive
    left_negative = left_negative[:-1]
    left_positive = left_positive[:-1]
    right_negative = right_negative[:-1]
    right_positive = right_positive[:-1]


    # initialize total number of data
    n = len(y)
    left_negative = left_negative[valid]
    left_positive = left_positive[valid]
    right_negative = right_negative[valid]
    right_positive = right_positive[valid]

    left_count = left_negative + left_positive
    right_count = right_negative + right_positive
    
    left_gini = gini(left_negative, left_positive, left_count)
    right_gini = gini(right_negative, right_positive, right_count)
    
    weighted_gini = (left_count / n) *  left_gini + (right_count / n) * right_gini
    print('weighted gini',weighted_gini)

    best_gini = weighted_gini.min()
    best_gini_index = weighted_gini.argmin()
    reduced_x = (x[:-1][valid] + x[1:][valid]) / 2

    best_split_value = reduced_x[best_gini_index]

    return best_split_value, best_gini


In [217]:
split, best_gini = best_gini_CART_ready(df2)
print(f"Best gini {best_gini} and the threshold is {split}")

weighted gini [0.14814815 0.28571429 0.4       ]
Best gini 0.14814814814814814 and the threshold is 1.5
